In [11]:
## 1. import / 경로 / DB 연결
import sqlite3
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\취준\olist-ecommerce-analysis")

DB_PATH = PROJECT_DIR / "db" / "olist_ecommerce.db"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(exist_ok=True)

conn = sqlite3.connect(DB_PATH)

print("DB_PATH:", DB_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

DB_PATH: C:\취준\olist-ecommerce-analysis\db\olist_ecommerce.db
OUTPUT_DIR: C:\취준\olist-ecommerce-analysis\outputs


In [12]:
## 2. 분석용 base table 생성
create_base_tables_sql = """
/*
2단계 목적:
- order_payments, order_reviews처럼 order_id 기준 중복 가능성이 있는 테이블을 먼저 주문 단위로 집계한다.
- delivered 주문 기준 주문 단위 base table과 상품 단위 base table을 분리한다.
- 이후 월별 KPI, 카테고리 매출, 배송 지연, 리뷰 점수 분석에 활용한다.
*/

DROP TABLE IF EXISTS payment_by_order;
DROP TABLE IF EXISTS review_by_order;
DROP TABLE IF EXISTS order_base_delivered;
DROP TABLE IF EXISTS order_item_base_delivered;

/*
1. 주문 단위 결제금액 테이블

주의:
order_payments는 한 order_id에 여러 결제 row가 존재할 수 있다.
따라서 매출 분석 전 order_id 기준으로 payment_value를 합산한다.
*/
CREATE TABLE payment_by_order AS
SELECT
    order_id,
    SUM(payment_value) AS payment_value,
    COUNT(*) AS payment_row_count,
    COUNT(DISTINCT payment_type) AS payment_type_count
FROM order_payments
GROUP BY order_id;

/*
2. 주문 단위 리뷰점수 테이블

주의:
order_reviews도 order_id 기준으로 여러 row가 존재할 수 있다.
본 프로젝트에서는 주문 단위 평균 리뷰 점수를 사용한다.
*/
CREATE TABLE review_by_order AS
SELECT
    order_id,
    AVG(review_score) AS review_score,
    COUNT(*) AS review_row_count
FROM order_reviews
GROUP BY order_id;

/*
3. delivered 주문 기준 주문 단위 base table

분석 단위:
- 1행 = 배송 완료 주문 1건

활용:
- 월별 주문 수
- 월별 매출
- 객단가
- 배송 지연율
- 고객 지역별 배송 지연율
- 배송 지연과 리뷰 점수 관계
*/
CREATE TABLE order_base_delivered AS
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    o.order_status,
    o.order_purchase_timestamp,
    o.purchase_date,
    o.purchase_year,
    o.purchase_month,
    o.purchase_dayofweek,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    o.delivery_days,
    o.delay_days,
    o.is_delayed,
    p.payment_value,
    p.payment_row_count,
    r.review_score,
    r.review_row_count
FROM orders_enriched o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
LEFT JOIN payment_by_order p
    ON o.order_id = p.order_id
LEFT JOIN review_by_order r
    ON o.order_id = r.order_id
WHERE o.order_status = 'delivered';

/*
4. delivered 주문 기준 상품 단위 base table

분석 단위:
- 1행 = 배송 완료 주문에 포함된 상품 row 1건

활용:
- 카테고리별 매출
- 카테고리별 주문 수
- 카테고리별 배송 지연율
- 판매자 지역 분석

주의:
order_items는 주문 1건에 여러 상품이 있을 수 있다.
따라서 이 테이블에서는 order_id가 중복되는 것이 정상이다.
*/
CREATE TABLE order_item_base_delivered AS
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    o.purchase_year,
    o.purchase_month,
    o.delivery_days,
    o.delay_days,
    o.is_delayed,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.price,
    oi.freight_value,
    COALESCE(
        t.product_category_name_english,
        pr.product_category_name,
        'unknown'
    ) AS product_category,
    s.seller_city,
    s.seller_state,
    r.review_score
FROM orders_enriched o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
INNER JOIN order_items oi
    ON o.order_id = oi.order_id
LEFT JOIN products pr
    ON oi.product_id = pr.product_id
LEFT JOIN category_translation t
    ON pr.product_category_name = t.product_category_name
LEFT JOIN sellers s
    ON oi.seller_id = s.seller_id
LEFT JOIN review_by_order r
    ON o.order_id = r.order_id
WHERE o.order_status = 'delivered';
"""

conn.executescript(create_base_tables_sql)
conn.commit()

print("분석용 base table 생성 완료")

분석용 base table 생성 완료


In [13]:
## 3. base table 검증
base_table_check_query = """
SELECT
    'payment_by_order' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT order_id) AS distinct_order_count
FROM payment_by_order

UNION ALL

SELECT
    'review_by_order' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT order_id) AS distinct_order_count
FROM review_by_order

UNION ALL

SELECT
    'order_base_delivered' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT order_id) AS distinct_order_count
FROM order_base_delivered

UNION ALL

SELECT
    'order_item_base_delivered' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT order_id) AS distinct_order_count
FROM order_item_base_delivered;
"""

base_table_check = pd.read_sql_query(base_table_check_query, conn)
display(base_table_check)

order_base_sample = pd.read_sql_query(
    """
    SELECT *
    FROM order_base_delivered
    LIMIT 10;
    """,
    conn
)

display(order_base_sample)

order_item_base_sample = pd.read_sql_query(
    """
    SELECT *
    FROM order_item_base_delivered
    LIMIT 10;
    """,
    conn
)

display(order_item_base_sample)

,table_name,row_count,distinct_order_count
0,payment_by_order,99440,99440
1,review_by_order,98673,98673
2,order_base_delivered,96478,96478
3,order_item_base_delivered,110197,96478


,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp,purchase_date,purchase_year,purchase_month,...,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delay_days,is_delayed,payment_value,payment_row_count,review_score,review_row_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017-10-02 10:56:33,2017-10-02,2017,2017-10,...,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,8.436574,-7.107488,0,38.71,3,4.0,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018-07-24 20:41:37,2018-07-24,2018,2018-07,...,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,13.782037,-5.355729,0,141.46,1,4.0,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018-08-08 08:38:49,2018-08-08,2018,2018-08,...,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,9.394213,-17.245498,0,179.12,1,5.0,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,delivered,2017-11-18 19:28:06,2017-11-18,2017,2017-11,...,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,13.208750,-12.980069,0,72.20,1,5.0,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018-02-13 21:18:39,2018-02-13,2018,2018-02,...,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2.873877,-9.238171,0,28.62,1,5.0,1
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2,congonhinhas,PR,delivered,2017-07-09 21:57:05,2017-07-09,2017,2017-07,...,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00,16.542245,-5.543113,0,175.26,1,4.0,1
6,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,932afa1e708222e5821dac9cd5db4cae,nilopolis,RJ,delivered,2017-05-16 13:10:30,2017-05-16,2017,2017-05,...,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00,9.989826,-11.461215,0,75.16,1,5.0,1
7,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,39382392765b6dc74812866ee5ee92a7,faxinalzinho,RS,delivered,2017-01-23 18:29:09,2017-01-23,2017,2017-01,...,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00,9.818762,-31.410995,0,35.95,1,1.0,1
8,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,299905e3934e9e181bfb2e164dd4b4f8,sorocaba,SP,delivered,2017-07-29 11:55:02,2017-07-29,2017,2017-07,...,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00,18.221852,-6.281597,0,169.76,2,5.0,1
9,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,f2a85dec752b8517b5e58a06ff3cd937,rio de janeiro,RJ,delivered,2017-05-16 19:41:10,2017-05-16,2017,2017-05,...,2017-05-18 11:40:40,2017-05-29 11:18:31,2017-06-07 00:00:00,12.650937,-8.528808,0,259.06,1,1.0,1


,order_id,customer_id,customer_unique_id,customer_city,customer_state,purchase_year,purchase_month,delivery_days,delay_days,is_delayed,order_item_id,product_id,seller_id,price,freight_value,product_category,seller_city,seller_state,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,2017,2017-10,8.436574,-7.107488,0,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,housewares,maua,SP,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,2018,2018-07,13.782037,-5.355729,0,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,perfumery,belo horizonte,SP,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,2018,2018-08,9.394213,-17.245498,0,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,auto,guariba,SP,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,2017,2017-11,13.208750,-12.980069,0,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,pet_shop,belo horizonte,MG,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,2018,2018-02,2.873877,-9.238171,0,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,stationery,mogi das cruzes,SP,5.0
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2,congonhinhas,PR,2017,2017-07,16.542245,-5.543113,0,1,060cb19345d90064d1015407193c233d,8581055ce74af1daba164fdbd55a40de,147.90,27.36,auto,guarulhos,SP,4.0
6,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,932afa1e708222e5821dac9cd5db4cae,nilopolis,RJ,2017,2017-05,9.989826,-11.461215,0,1,4520766ec412348b8d4caa5e8a18c464,16090f2ca825584b5a147ab24aa30c86,59.99,15.17,auto,atibaia,SP,5.0
7,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,39382392765b6dc74812866ee5ee92a7,faxinalzinho,RS,2017,2017-01,9.818762,-31.410995,0,1,ac1789e492dcd698c5c10b97a671243a,63b9ae557efed31d1f7687917d248a8d,19.90,16.05,furniture_decor,sao jose do rio pardo,SP,1.0
8,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,299905e3934e9e181bfb2e164dd4b4f8,sorocaba,SP,2017,2017-07,18.221852,-6.281597,0,1,9a78fb9862b10749a117f7fc3c31f051,7c67e1448b00f6e969d365cea6b010ab,149.99,19.77,office_furniture,itaquaquecetuba,SP,5.0
9,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,f2a85dec752b8517b5e58a06ff3cd937,rio de janeiro,RJ,2017,2017-05,12.650937,-8.528808,0,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,99.00,30.53,garden_tools,cariacica,ES,1.0


In [14]:
## 4. 포트폴리오용 SQL 분석 5개 실행
# 4-1. 월별 주문 수·매출·객단가 추이
monthly_kpi_query = """
/*
분석 질문:
월별 주문 수, 매출, 객단가는 어떻게 변화했는가?

분석 단위:
- 1행 = 배송 완료 주문 1건

주요 지표:
- order_count: 월별 주문 수
- customer_count: 월별 고유 고객 수
- revenue: 월별 결제금액 합계
- avg_order_value: 객단가
- delayed_order_rate_pct: 배송 지연율

해석 주의:
payment_value는 주문 단위로 먼저 합산한 payment_by_order를 사용했다.
따라서 결제 row 중복으로 인한 매출 과대 집계를 방지했다.
*/

SELECT
    purchase_month,
    COUNT(DISTINCT order_id) AS order_count,
    COUNT(DISTINCT customer_unique_id) AS customer_count,
    ROUND(SUM(payment_value), 2) AS revenue,
    ROUND(SUM(payment_value) / COUNT(DISTINCT order_id), 2) AS avg_order_value,
    ROUND(AVG(delivery_days), 2) AS avg_delivery_days,
    ROUND(AVG(is_delayed) * 100, 2) AS delayed_order_rate_pct,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM order_base_delivered
GROUP BY purchase_month
ORDER BY purchase_month;
"""

monthly_kpi = pd.read_sql_query(monthly_kpi_query, conn)
display(monthly_kpi.head())

,purchase_month,order_count,customer_count,revenue,avg_order_value,avg_delivery_days,delayed_order_rate_pct,avg_review_score
0,2016-09,1,1,NaN,NaN,54.81,100.00,1.00
1,2016-10,265,262,46566.71,175.72,19.60,1.13,4.01
2,2016-12,1,1,19.62,19.62,4.69,0.00,5.00
3,2017-01,750,718,127545.67,170.06,12.65,3.07,4.20
4,2017-02,1653,1630,271298.65,164.13,13.17,3.21,4.20


In [15]:
## 4. 포트폴리오용 SQL 분석 5개 실행
# 4-2. 카테고리별 매출 TOP10
category_revenue_query = """
/*
분석 질문:
어떤 상품 카테고리가 매출에 가장 크게 기여했는가?

분석 단위:
- 1행 = 배송 완료 주문의 상품 row 1건

주요 지표:
- revenue: 상품 가격 합계
- freight_value: 배송비 합계
- order_count: 해당 카테고리를 포함한 주문 수
- item_count: 판매 상품 row 수
- avg_item_price: 평균 상품 가격

해석 주의:
이 분석의 revenue는 order_items.price 기준 상품 매출이다.
order_payments.payment_value 기준 결제금액과는 쿠폰, 배송비, 결제 구조 때문에 차이가 날 수 있다.
*/

SELECT
    product_category,
    COUNT(DISTINCT order_id) AS order_count,
    COUNT(*) AS item_count,
    ROUND(SUM(price), 2) AS revenue,
    ROUND(SUM(freight_value), 2) AS freight_revenue,
    ROUND(AVG(price), 2) AS avg_item_price,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM order_item_base_delivered
GROUP BY product_category
ORDER BY revenue DESC
LIMIT 10;
"""

category_revenue_top10 = pd.read_sql_query(category_revenue_query, conn)
display(category_revenue_top10)

,product_category,order_count,item_count,revenue,freight_revenue,avg_item_price,avg_review_score
0,health_beauty,8647,9465,1233131.72,178957.81,130.28,4.19
1,watches_gifts,5495,5859,1166176.98,98156.14,199.04,4.07
2,bed_bath_table,9272,10953,1023434.76,201774.50,93.44,3.92
3,sports_leisure,7530,8431,954852.55,163404.36,113.25,4.17
4,computers_accessories,6530,7644,888724.61,143999.16,116.26,3.99
5,furniture_decor,6307,8160,711927.69,168402.23,87.25,3.95
6,housewares,5743,6795,615628.69,142763.56,90.60,4.11
7,cool_stuff,3559,3718,610204.10,81476.79,164.12,4.19
8,auto,3810,4140,578966.65,90488.10,139.85,4.12
9,toys,3804,4030,471286.48,75774.58,116.94,4.21


In [16]:
## 4. 포트폴리오용 SQL 분석 5개 실행
# 4-3. Window Function 기반 카테고리 매출 비중
category_revenue_share_query = """
/*
분석 질문:
카테고리별 매출 비중과 누적 매출 비중은 어떻게 나타나는가?

활용 SQL:
- SUM() OVER()
- SUM() OVER(ORDER BY ...)
- Window Function

포트폴리오 포인트:
단순 GROUP BY뿐 아니라 Window Function으로 전체 대비 비중과 누적 비중을 계산했다.
*/

WITH category_sales AS (
    SELECT
        product_category,
        COUNT(DISTINCT order_id) AS order_count,
        COUNT(*) AS item_count,
        SUM(price) AS revenue
    FROM order_item_base_delivered
    GROUP BY product_category
),

category_share AS (
    SELECT
        product_category,
        order_count,
        item_count,
        revenue,
        revenue / SUM(revenue) OVER() AS revenue_share,
        SUM(revenue) OVER(
            ORDER BY revenue DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) / SUM(revenue) OVER() AS cumulative_revenue_share,
        RANK() OVER(ORDER BY revenue DESC) AS revenue_rank
    FROM category_sales
)

SELECT
    revenue_rank,
    product_category,
    order_count,
    item_count,
    ROUND(revenue, 2) AS revenue,
    ROUND(revenue_share * 100, 2) AS revenue_share_pct,
    ROUND(cumulative_revenue_share * 100, 2) AS cumulative_revenue_share_pct
FROM category_share
ORDER BY revenue_rank
LIMIT 20;
"""

category_revenue_share = pd.read_sql_query(category_revenue_share_query, conn)
display(category_revenue_share)

,revenue_rank,product_category,order_count,item_count,revenue,revenue_share_pct,cumulative_revenue_share_pct
0,1,health_beauty,8647,9465,1233131.72,9.33,9.33
1,2,watches_gifts,5495,5859,1166176.98,8.82,18.15
2,3,bed_bath_table,9272,10953,1023434.76,7.74,25.89
3,4,sports_leisure,7530,8431,954852.55,7.22,33.11
4,5,computers_accessories,6530,7644,888724.61,6.72,39.83
5,6,furniture_decor,6307,8160,711927.69,5.38,45.22
6,7,housewares,5743,6795,615628.69,4.66,49.87
7,8,cool_stuff,3559,3718,610204.10,4.62,54.49
8,9,auto,3810,4140,578966.65,4.38,58.87
9,10,toys,3804,4030,471286.48,3.56,62.43


In [17]:
## 4. 포트폴리오용 SQL 분석 5개 실행
# 4-4. 고객 지역별 배송 지연율
delivery_delay_by_state_query = """
/*
분석 질문:
고객 지역별 배송 지연율은 어떻게 다른가?

분석 단위:
- 1행 = 배송 완료 주문 1건

해석 주의:
주문 수가 너무 적은 지역은 지연율이 왜곡될 수 있으므로,
주문 수 100건 이상인 주만 비교한다.
*/

SELECT
    customer_state,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(AVG(delivery_days), 2) AS avg_delivery_days,
    ROUND(AVG(delay_days), 2) AS avg_delay_days,
    SUM(is_delayed) AS delayed_order_count,
    ROUND(AVG(is_delayed) * 100, 2) AS delayed_order_rate_pct,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM order_base_delivered
GROUP BY customer_state
HAVING COUNT(DISTINCT order_id) >= 100
ORDER BY delayed_order_rate_pct DESC;
"""

delivery_delay_by_state = pd.read_sql_query(delivery_delay_by_state_query, conn)
display(delivery_delay_by_state.head(10))

,customer_state,order_count,avg_delivery_days,avg_delay_days,delayed_order_count,delayed_order_rate_pct,avg_review_score
0,AL,397,24.54,-8.03,95,23.93,3.85
1,MA,717,21.57,-8.89,141,19.67,3.83
2,PI,476,19.46,-10.63,76,15.97,3.99
3,CE,1279,21.27,-10.11,196,15.32,3.94
4,SE,335,21.52,-9.33,51,15.22,3.91
5,BA,3256,19.34,-10.10,457,14.04,3.93
6,RJ,12350,15.31,-11.05,1664,13.47,3.97
7,TO,274,17.66,-11.44,35,12.77,4.15
8,PA,946,23.77,-13.39,117,12.37,3.91
9,ES,1995,15.79,-9.80,244,12.23,4.08


In [18]:
## 4. 포트폴리오용 SQL 분석 5개 실행
# 4-5. 배송 지연 구간별 평균 리뷰 점수
review_score_by_delay_group_query = """
/*
분석 질문:
배송 지연 정도에 따라 평균 리뷰 점수는 어떻게 달라지는가?

분석 단위:
- 1행 = 배송 완료 주문 1건

지연 구간 정의:
- early_7plus_days: 예상일보다 7일 이상 빨리 도착
- early_1_to_6_days: 예상일보다 1~6일 빨리 도착
- on_time: 예상일 당일 도착
- delayed_1_to_3_days: 1~3일 지연
- delayed_4_to_7_days: 4~7일 지연
- delayed_8plus_days: 8일 이상 지연

핵심 해석:
배송 지연이 커질수록 리뷰 점수가 낮아지는지 확인한다.
*/

WITH delay_grouped AS (
    SELECT
        order_id,
        review_score,
        delay_days,
        CASE
            WHEN delay_days <= -7 THEN 'early_7plus_days'
            WHEN delay_days < 0 THEN 'early_1_to_6_days'
            WHEN delay_days = 0 THEN 'on_time'
            WHEN delay_days BETWEEN 0.000001 AND 3 THEN 'delayed_1_to_3_days'
            WHEN delay_days > 3 AND delay_days <= 7 THEN 'delayed_4_to_7_days'
            WHEN delay_days > 7 THEN 'delayed_8plus_days'
            ELSE 'unknown'
        END AS delay_group,
        CASE
            WHEN delay_days <= -7 THEN 1
            WHEN delay_days < 0 THEN 2
            WHEN delay_days = 0 THEN 3
            WHEN delay_days BETWEEN 0.000001 AND 3 THEN 4
            WHEN delay_days > 3 AND delay_days <= 7 THEN 5
            WHEN delay_days > 7 THEN 6
            ELSE 99
        END AS delay_group_order
    FROM order_base_delivered
    WHERE review_score IS NOT NULL
      AND delay_days IS NOT NULL
)

SELECT
    delay_group_order,
    delay_group,
    COUNT(*) AS order_count,
    ROUND(AVG(delay_days), 2) AS avg_delay_days,
    ROUND(AVG(review_score), 2) AS avg_review_score,
    ROUND(SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS low_score_rate_pct,
    ROUND(SUM(CASE WHEN review_score = 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS five_star_rate_pct
FROM delay_grouped
GROUP BY delay_group_order, delay_group
ORDER BY delay_group_order;
"""

review_score_by_delay_group = pd.read_sql_query(review_score_by_delay_group_query, conn)
display(review_score_by_delay_group)

,delay_group_order,delay_group,order_count,avg_delay_days,avg_review_score,low_score_rate_pct,five_star_rate_pct
0,1,early_7plus_days,70929,-15.16,4.32,8.94,63.58
1,2,early_1_to_6_days,17234,-4.16,4.20,10.21,57.43
2,4,delayed_1_to_3_days,2636,1.40,3.77,19.08,43.13
3,5,delayed_4_to_7_days,1773,5.11,2.32,61.25,18.10
4,6,delayed_8plus_days,3252,18.33,1.73,78.32,7.47


In [19]:
## 5. 결과 CSV 저장 / DB 연결 종료
# SQL 분석 결과 저장

monthly_kpi.to_csv(
    OUTPUT_DIR / "monthly_kpi.csv",
    index=False,
    encoding="utf-8-sig"
)

category_revenue_top10.to_csv(
    OUTPUT_DIR / "category_revenue_top10.csv",
    index=False,
    encoding="utf-8-sig"
)

category_revenue_share.to_csv(
    OUTPUT_DIR / "category_revenue_share.csv",
    index=False,
    encoding="utf-8-sig"
)

delivery_delay_by_state.to_csv(
    OUTPUT_DIR / "delivery_delay_by_state.csv",
    index=False,
    encoding="utf-8-sig"
)

review_score_by_delay_group.to_csv(
    OUTPUT_DIR / "review_score_by_delay_group.csv",
    index=False,
    encoding="utf-8-sig"
)

# SQL 쿼리 저장용 텍스트 파일 생성
sql_queries = {
    "00_create_base_tables.sql": create_base_tables_sql,
    "01_monthly_kpi.sql": monthly_kpi_query,
    "02_category_revenue_top10.sql": category_revenue_query,
    "03_category_revenue_share_window_function.sql": category_revenue_share_query,
    "04_delivery_delay_by_state.sql": delivery_delay_by_state_query,
    "05_review_score_by_delay_group.sql": review_score_by_delay_group_query,
}

SQL_DIR = PROJECT_DIR / "sql"
SQL_DIR.mkdir(exist_ok=True)

for file_name, query in sql_queries.items():
    with open(SQL_DIR / file_name, "w", encoding="utf-8") as f:
        f.write(query)

print("분석 결과 CSV 저장 완료")
print("SQL 쿼리 파일 저장 완료")

conn.close()
print("SQLite 연결 종료 완료")

분석 결과 CSV 저장 완료
SQL 쿼리 파일 저장 완료
SQLite 연결 종료 완료


#### ★ DB 저장 실패 확인 -> 개선